# Trabajo Práctico Final — Sistema de Detección y Segmentación de Objetos en Tiempo Real
## Notebook 02 · Entrenamiento del modelo (fine-tuning de YOLOv8n-seg)

**Maestría en Inteligencia Artificial — Deep Learning**

---

Este notebook implementa la segunda etapa del pipeline: el **ajuste fino (fine-tuning)** de un modelo de segmentación de instancias **YOLOv8n-seg**, preentrenado sobre COCO, para las ocho clases definidas en el Notebook 01. Se documentan la arquitectura, el esquema de preprocesamiento y aumento de datos (*data augmentation*), la configuración de entrenamiento y el análisis de las curvas de aprendizaje.

> **Nota de ejecución:** este notebook debe ejecutarse en el **servidor con GPU** (JupyterLab), luego de haber ejecutado el Notebook 01 en el mismo entorno. Requiere que exista `data/coco_subset/dataset.yaml`.

### 1. Verificación del entorno de cómputo

Se comprueba la disponibilidad de la GPU y la versión de las bibliotecas. El entrenamiento de redes convolucionales profundas sobre miles de imágenes es computacionalmente intensivo; la aceleración por GPU reduce el tiempo de entrenamiento de horas (CPU) a minutos.

In [1]:
import torch
import ultralytics

print('PyTorch:', torch.__version__)
print('Ultralytics:', ultralytics.__version__)
print('CUDA disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print(f'Memoria total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

PyTorch: 2.12.1+cu130
Ultralytics: 8.4.89
CUDA disponible: True
GPU: NVIDIA GeForce RTX 5090
Memoria total: 33.7 GB


In [2]:
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from ultralytics import YOLO

SEED = 42

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
DATA_YAML = ROOT / 'data' / 'coco_subset' / 'dataset.yaml'
RUNS_DIR = ROOT / 'runs'
MODELS_DIR = ROOT / 'models'
MODELS_DIR.mkdir(exist_ok=True)

assert DATA_YAML.exists(), 'No se encontró dataset.yaml: ejecutar primero el Notebook 01 en este entorno.'

# Estilo de los gráficos
C_AZUL, C_AQUA = '#2a78d6', '#1baf7a'
plt.rcParams.update({
    'figure.facecolor': '#fcfcfb', 'axes.facecolor': '#fcfcfb',
    'axes.edgecolor': '#c3c2b7', 'axes.grid': True, 'axes.axisbelow': True,
    'grid.color': '#e1e0d9', 'grid.linewidth': 0.8,
    'axes.titlesize': 12, 'font.family': 'sans-serif',
})

print('Dataset:', DATA_YAML)

Dataset: /data/students/federico.moran/TPF - Computer Vision/data/coco_subset/dataset.yaml


### 2. Arquitectura y estrategia de transferencia

**YOLOv8-seg** es un modelo de segmentación de instancias de una sola etapa (*one-stage*) y libre de anclas (*anchor-free*), compuesto por:

- **Backbone** convolucional (CSPDarknet con bloques C2f) que extrae mapas de características a múltiples escalas.
- **Neck** de tipo PAN-FPN que fusiona información semántica de alto nivel con detalle espacial de bajo nivel.
- **Cabeza desacoplada** que predice, por cada objeto, la caja delimitadora, la clase y un vector de coeficientes de máscara; una rama adicional genera *prototipos* de máscara a resolución de imagen (esquema derivado de YOLACT). La máscara final de cada instancia se obtiene como combinación lineal de los prototipos, lo que permite segmentar en tiempo real.

Se adopta la variante **nano** (`yolov8n-seg`, ~3,4 M de parámetros) porque la demostración final se ejecuta sobre CPU en una computadora personal: es la variante con mejor relación desempeño/latencia para ese escenario.

**Estrategia de transferencia.** Se parte de los pesos preentrenados en COCO completo y se ajustan **todos** los parámetros sobre el subconjunto de 8 clases (*fine-tuning* completo, sin congelamiento de capas). Dado que el dominio de origen y el de destino coinciden (las 8 clases pertenecen a COCO), el preentrenamiento aporta representaciones ya adecuadas y el ajuste se concentra en especializar la cabeza de clasificación; esto permite converger en pocas épocas con un conjunto de datos moderado.

### 3. Preprocesamiento y aumento de datos

**Preprocesamiento (train e inferencia).** Toda imagen se redimensiona a 640×640 píxeles mediante *letterbox* (se preserva la relación de aspecto y se rellena con bordes neutros) y los valores de intensidad se normalizan al rango [0, 1].

**Aumento de datos (solo entrenamiento).** El aumento de datos amplía artificialmente la variabilidad del conjunto de entrenamiento y actúa como regularizador, reduciendo el sobreajuste. Se emplea el esquema estándar de Ultralytics, que transforma de manera consistente imágenes **y** polígonos de segmentación:

| Transformación | Valor | Justificación |
|---|---|---|
| *Mosaic* | 1.0 | Compone 4 imágenes en una: expone al modelo a objetos en contextos y escalas atípicos y aumenta la densidad de instancias por lote. Se desactiva en las últimas 10 épocas (`close_mosaic=10`) para que el modelo termine de ajustarse sobre imágenes de estadística natural. |
| Volteo horizontal | p = 0.5 | Los objetos del dominio (tazas, botellas, personas) no tienen quiralidad relevante; el volteo duplica la variabilidad de pose. |
| Traslación | ±10 % | Robustez ante encuadres imperfectos, frecuentes en una webcam. |
| Escala | ±50 % | Robustez ante variaciones de distancia objeto–cámara, críticas para la demostración en vivo. |
| Perturbación HSV | h=0.015, s=0.7, v=0.4 | Robustez ante cambios de iluminación y balance de color entre las escenas de COCO y la webcam del usuario. |
| Volteo vertical / rotación / *mixup* | 0 | Se omiten: generarían poses inverosímiles para el dominio (por ejemplo, personas invertidas) sin beneficio esperado. |

In [3]:
# Hiperparámetros de aumento de datos (explícitos para su documentación en el reporte)
AUG = dict(
    mosaic=1.0, close_mosaic=10,
    fliplr=0.5, flipud=0.0,
    degrees=0.0, translate=0.1, scale=0.5, shear=0.0,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    mixup=0.0,
)
AUG

{'mosaic': 1.0,
 'close_mosaic': 10,
 'fliplr': 0.5,
 'flipud': 0.0,
 'degrees': 0.0,
 'translate': 0.1,
 'scale': 0.5,
 'shear': 0.0,
 'hsv_h': 0.015,
 'hsv_s': 0.7,
 'hsv_v': 0.4,
 'mixup': 0.0}

### 4. Configuración y ejecución del entrenamiento

Configuración adoptada:

- **Épocas:** 60, con *early stopping* de paciencia 15 sobre la métrica de validación; el mejor punto de control (`best.pt`) se selecciona automáticamente según el desempeño en la partición de validación.
- **Tamaño de lote:** automático (`batch=-1`), que ajusta el lote a la memoria disponible de la GPU.
- **Optimizador y tasa de aprendizaje:** configuración automática de Ultralytics (SGD/AdamW con calendario de decaimiento), adecuada como punto de partida robusto para fine-tuning.
- **Semilla fija** para la reproducibilidad del experimento.

Durante el entrenamiento se reporta, por época, la pérdida sobre train y val (componentes de caja, segmentación, clasificación y DFL) y el mAP sobre la partición de validación.

In [4]:
model = YOLO('yolov8n-seg.pt')   # pesos preentrenados en COCO (80 clases)

resultados = model.train(
    data=str(DATA_YAML),
    epochs=60,
    imgsz=640,
    batch=32,
    workers=4, 
    device=0,
    seed=SEED,
    patience=15,         # early stopping
    project=str(RUNS_DIR),
    name='yolov8n_seg_tpf',
    exist_ok=True,
    **AUG,
)

Ultralytics 8.4.89 🚀 Python-3.12.3 torch-2.12.1+cu130 CUDA:0 (NVIDIA GeForce RTX 5090, 32108MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/data/students/federico.moran/TPF - Computer Vision/data/coco_subset/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=60, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_seg_tpf, nbs=64, nms=Fa

### 5. Curvas de aprendizaje

Se analizan dos aspectos:

1. **Convergencia de las pérdidas.** Las pérdidas de caja y de segmentación deben decrecer de forma sostenida. Una brecha creciente entre la pérdida de entrenamiento y la de validación indicaría sobreajuste; el aumento de datos y el early stopping son las salvaguardas adoptadas.
2. **Evolución del mAP en validación.** Permite identificar la época a partir de la cual el desempeño se estabiliza. El repunte habitual en las últimas 10 épocas coincide con la desactivación de *mosaic* (`close_mosaic`), cuando el modelo se ajusta sobre imágenes de estadística natural.

In [8]:
res_csv = RUNS_DIR / 'yolov8n_seg_tpf' / 'results.csv'
df = pd.read_csv(res_csv)
df.columns = df.columns.str.strip()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Pérdidas de caja y segmentación (train vs val)
axes[0].plot(df['epoch'], df['train/box_loss'], color=C_AZUL, label='caja — train')
axes[0].plot(df['epoch'], df['val/box_loss'], color=C_AZUL, linestyle='--', label='caja — val')
axes[0].plot(df['epoch'], df['train/seg_loss'], color=C_AQUA, label='segmentación — train')
axes[0].plot(df['epoch'], df['val/seg_loss'], color=C_AQUA, linestyle='--', label='segmentación — val')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Pérdida')
axes[0].set_title('Evolución de las pérdidas')
axes[0].legend()

# mAP en validación
axes[1].plot(df['epoch'], df['metrics/mAP50(B)'], color=C_AZUL, label='mAP@50 — cajas')
axes[1].plot(df['epoch'], df['metrics/mAP50(M)'], color=C_AQUA, label='mAP@50 — máscaras')
axes[1].plot(df['epoch'], df['metrics/mAP50-95(B)'], color=C_AZUL, linestyle='--', label='mAP@50-95 — cajas')
axes[1].plot(df['epoch'], df['metrics/mAP50-95(M)'], color=C_AQUA, linestyle='--', label='mAP@50-95 — máscaras')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('mAP (validación)')
axes[1].set_title('Evolución del mAP en validación')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'Mejor mAP@50-95 de máscaras en validación: {df["metrics/mAP50-95(M)"].max():.3f} (época {df.loc[df["metrics/mAP50-95(M)"].idxmax(), "epoch"]:.0f})')

<Figure size 1300x450 with 2 Axes>

Mejor mAP@50-95 de máscaras en validación: 0.450 (época 60)


### 6. Consolidación del modelo entrenado

Se copia el mejor punto de control (`best.pt`, seleccionado por mAP de validación) al directorio `models/` del proyecto. Este archivo constituye uno de los **entregables** del trabajo y es el que consumen el Notebook 03 (evaluación) y el script de demostración en tiempo real (`src/demo_webcam.py`).

In [6]:
best = RUNS_DIR / 'yolov8n_seg_tpf' / 'weights' / 'best.pt'
destino = MODELS_DIR / 'yolov8n_seg_best.pt'
shutil.copy2(best, destino)
print('Pesos consolidados en:', destino)
print(f'Tamaño: {destino.stat().st_size / 1e6:.1f} MB')

Pesos consolidados en: /data/students/federico.moran/TPF - Computer Vision/models/yolov8n_seg_best.pt
Tamaño: 6.8 MB


### 7. Experimento complementario (opcional): capacidad del modelo

Para cuantificar el compromiso entre capacidad y latencia se puede entrenar, con configuración idéntica, la variante **small** (`yolov8s-seg`, ~11,8 M de parámetros). La comparación de mAP y de velocidad de inferencia entre ambas variantes se incorpora en el Notebook 03 y en el reporte técnico como análisis del compromiso precisión–latencia, relevante porque el sistema final debe operar en tiempo real sobre CPU.

Para ejecutarlo, cambiar `TRAIN_YOLOV8S = True`.

In [1]:
TRAIN_YOLOV8S = False

if TRAIN_YOLOV8S:
    model_s = YOLO('yolov8s-seg.pt')
    model_s.train(
        data=str(DATA_YAML),
        epochs=60,
        imgsz=640,
        batch=-1,
        device=0,
        seed=SEED,
        patience=15,
        project=str(RUNS_DIR),
        name='yolov8s_seg_tpf',
        exist_ok=True,
        **AUG,
    )
    shutil.copy2(RUNS_DIR / 'yolov8s_seg_tpf' / 'weights' / 'best.pt',
                 MODELS_DIR / 'yolov8s_seg_best.pt')
    print('Variante small entrenada y consolidada.')
else:
    print('Experimento omitido (TRAIN_YOLOV8S = False).')

Experimento omitido (TRAIN_YOLOV8S = False).


### 8. Síntesis y próximos pasos

Se realizó el ajuste fino de YOLOv8n-seg sobre el subconjunto de 8 clases, con aumento de datos documentado, early stopping y selección automática del mejor punto de control. Los pesos finales quedaron consolidados en `models/yolov8n_seg_best.pt`.

El **Notebook 03** evalúa este modelo sobre la partición de **prueba** (no utilizada durante el entrenamiento), reporta las métricas estándar (mAP@50, mAP@50-95, para cajas y máscaras), analiza la matriz de confusión y lo compara cuantitativa y cualitativamente con el modelo preentrenado de referencia.